# COSC 2671 — Assignment 2
## Notebook 2: Data Cleaning
Flatten raw JSON → clean CSVs ready for network and NLP analysis.
One output CSV per repo, containing both issues and comments as rows.

***Cell 2 — Set Working Directory***

In [1]:
from pathlib import Path
import os

PROJECT_ROOT = next(Path.home().rglob("data/raw")).parent.parent
os.chdir(PROJECT_ROOT)

print(f"Working directory: {os.getcwd()}")
print(f"data/raw exists: {os.path.exists('data/raw')}")

Working directory: /Users/rob/Desktop/2026/social/SMNA2026-A2-github-networks
data/raw exists: True


***Cell 3 Imports***


In [2]:
import re
import json
import pandas as pd

RAW_DIR  = "data/raw"
PROC_DIR = "data/processed"
os.makedirs(PROC_DIR,         exist_ok=True)
os.makedirs("outputs/tables", exist_ok=True)

REPOS = [
    "scikit-learn/scikit-learn",
    "huggingface/transformers",
    "pytorch/pytorch",
    "keras-team/keras",
    "langchain-ai/langchain"
]

SAFE_NAMES = {
    "scikit-learn/scikit-learn":  "scikit-learn_scikit-learn",
    "huggingface/transformers":   "huggingface_transformers",
    "pytorch/pytorch":            "pytorch_pytorch",
    "keras-team/keras":           "keras-team_keras",
    "langchain-ai/langchain":     "langchain-ai_langchain"
}

print("Imports OK.")

Imports OK.


***Cell 4 Text Cleaning Function***


In [3]:
def clean_text(text):
    """
    Clean raw GitHub issue/comment text for NLP.
    Removes: code blocks, URLs, markdown, HTML, mentions, excess whitespace.
    """
    if not text or not isinstance(text, str):
        return ""

    # Remove fenced code blocks
    text = re.sub(r"```[\s\S]*?```", " ", text)
    # Remove inline code
    text = re.sub(r"`[^`]*`", " ", text)
    # Remove URLs
    text = re.sub(r"https?://\S+", " ", text)
    # Remove markdown headers
    text = re.sub(r"#+\s", " ", text)
    # Remove bold/italic
    text = re.sub(r"[*_]{1,2}([^*_]+)[*_]{1,2}", r"\1", text)
    # Remove HTML tags
    text = re.sub(r"<[^>]+>", " ", text)
    # Remove GitHub @mentions
    text = re.sub(r"@\w+", " ", text)
    # Remove image and link markdown
    text = re.sub(r"!\[.*?\]\(.*?\)", " ", text)
    text = re.sub(r"\[.*?\]\(.*?\)", " ", text)
    # Remove special characters except basic punctuation
    text = re.sub(r"[^\w\s.,!?;:()\-']", " ", text)
    # Collapse whitespace
    text = re.sub(r"\s+", " ", text).strip()

    return text.lower()

***Cell 4b Secret Redaction Function***


In [4]:
SECRET_PATTERNS = [
    # OpenAI / OpenRouter keys  (sk-...)
    (re.compile(r'sk-[A-Za-z0-9]{20,}'),                               '[REDACTED_API_KEY]'),
    # AWS Access Key IDs  (AKIA...)
    (re.compile(r'AKIA[A-Z0-9]{16}'),                                   '[REDACTED_AWS_KEY]'),
    # AWS Secret Access Keys  (40-char base64-ish)
    (re.compile(r'(?<![A-Za-z0-9/+])[A-Za-z0-9/+]{40}(?![A-Za-z0-9/+])'), '[REDACTED_AWS_SECRET]'),
    # Generic Bearer tokens
    (re.compile(r'(?i)(bearer\s+)[A-Za-z0-9\-._~+/]{20,}'),            r'\1[REDACTED_TOKEN]'),
    # GitHub classic PATs  (ghp_ / ghs_)
    (re.compile(r'gh[ps]_[A-Za-z0-9]{36,}'),                           '[REDACTED_GH_TOKEN]'),
    # GitHub fine-grained PATs
    (re.compile(r'github_pat_[A-Za-z0-9_]{82}'),                       '[REDACTED_GH_TOKEN]'),
]

def redact_secrets(text):
    """Replace secret-looking values with safe placeholders."""
    if not text or not isinstance(text, str):
        return text
    for pattern, replacement in SECRET_PATTERNS:
        text = pattern.sub(replacement, text)
    return text

print("redact_secrets() ready — patterns loaded:", len(SECRET_PATTERNS))

redact_secrets() ready — patterns loaded: 6


***Cell 5 - Flatten Function***

In [5]:
def flatten_repo(filepath):
    """
    Load a raw JSON file and flatten into a single DataFrame.
    Each row is either an issue or a comment.

    Columns:
      id, type, author, text, clean_text, state, created_at, closed_at,
      parent_id, parent_author, labels, issue_number, comment_count
    """
    with open(filepath, encoding="utf-8") as f:
        issues = json.load(f)

    rows = []

    for issue in issues:
        issue_id     = f"issue_{issue['issue_number']}"
        issue_author = issue.get("author")
        raw_text     = (issue.get("title") or "") + " " + (issue.get("body") or "")

        rows.append({
            "id":            issue_id,
            "type":          "issue",
            "author":        issue_author,
            "text":          raw_text[:5000],
            "clean_text":    clean_text(raw_text),
            "state":         issue.get("state"),
            "created_at":    issue.get("created_at"),
            "closed_at":     issue.get("closed_at"),
            "parent_id":     None,
            "parent_author": None,
            "labels":        ", ".join(issue.get("labels", [])),
            "issue_number":  issue["issue_number"],
            "comment_count": issue.get("comment_count", 0)
        })

        for comment in issue.get("comments", []):
            comment_body = comment.get("body") or ""
            rows.append({
                "id":            f"comment_{comment['comment_id']}",
                "type":          "comment",
                "author":        comment.get("author"),
                "text":          comment_body[:3000],
                "clean_text":    clean_text(comment_body),
                "state":         issue.get("state"),
                "created_at":    comment.get("created_at"),
                "closed_at":     None,
                "parent_id":     issue_id,
                "parent_author": issue_author,
                "labels":        "",
                "issue_number":  issue["issue_number"],
                "comment_count": 0
            })

    df = pd.DataFrame(rows)
    df["created_at"] = pd.to_datetime(df["created_at"], errors="coerce", utc=True)
    return df

***Cell 6 Run Cleaning***

In [6]:
repo_stats = []

for repo_name in REPOS:
    safe_name   = SAFE_NAMES[repo_name]
    input_path  = f"{RAW_DIR}/{safe_name}_issues.json"
    output_path = f"{PROC_DIR}/{safe_name}_clean.csv"

    if not os.path.exists(input_path):
        print(f"⚠️  Skipping {repo_name} — raw file not found")
        continue

    print(f"Cleaning: {repo_name} ...", end=" ")
    df = flatten_repo(input_path)

    # Drop rows with no author
    df = df[df["author"].notna()].copy()
    # Drop rows with empty clean text
    df = df[df["clean_text"].str.len() > 10].copy()

    df.to_csv(output_path, index=False)

    issues_df   = df[df["type"] == "issue"]
    comments_df = df[df["type"] == "comment"]

    stats = {
        "repo":          repo_name,
        "total_rows":    len(df),
        "issues":        len(issues_df),
        "comments":      len(comments_df),
        "unique_users":  df["author"].nunique(),
        "open_issues":   (issues_df["state"] == "open").sum(),
        "closed_issues": (issues_df["state"] == "closed").sum(),
        "date_min":      str(df["created_at"].min())[:10],
        "date_max":      str(df["created_at"].max())[:10],
    }
    repo_stats.append(stats)
    print(f"✅  {len(df):,} rows → {output_path}")

stats_df = pd.DataFrame(repo_stats)
stats_df.to_csv("outputs/tables/data_summary.csv", index=False)
print("\n\nDATA SUMMARY TABLE")
print(stats_df.to_string(index=False))

Cleaning: scikit-learn/scikit-learn ... ✅  11,924 rows → data/processed/scikit-learn_scikit-learn_clean.csv
Cleaning: huggingface/transformers ... ✅  9,881 rows → data/processed/huggingface_transformers_clean.csv
Cleaning: pytorch/pytorch ... ✅  6,746 rows → data/processed/pytorch_pytorch_clean.csv
Cleaning: keras-team/keras ... ✅  11,205 rows → data/processed/keras-team_keras_clean.csv
Cleaning: langchain-ai/langchain ... ✅  8,323 rows → data/processed/langchain-ai_langchain_clean.csv


DATA SUMMARY TABLE
                     repo  total_rows  issues  comments  unique_users  open_issues  closed_issues   date_min   date_max
scikit-learn/scikit-learn       11924    1996      9928          1860          428           1568 2023-04-23 2026-05-13
 huggingface/transformers        9881    1993      7888          2378          250           1743 2025-04-02 2026-05-14
          pytorch/pytorch        6746    1998      4748           907          938           1060 2026-02-04 2026-05-14
        

***Cell 7 — Bot Detection***

In [7]:
BOT_PATTERNS = ["bot", "-bot", "github-actions", "dependabot", "stale", "codecov"]

print("Checking for bots in each repo:\n")
all_bots = set()

for repo_name in REPOS:
    safe_name = SAFE_NAMES[repo_name]
    path      = f"data/processed/{safe_name}_clean.csv"

    if not os.path.exists(path):
        print(f"  ⚠️  File not found: {path}")
        continue

    df    = pd.read_csv(path)
    users = df["author"].dropna().unique()
    bots  = [u for u in users if any(p.lower() in u.lower() for p in BOT_PATTERNS)]

    if bots:
        print(f"  {repo_name}")
        for b in sorted(bots):
            count = (df["author"] == b).sum()
            print(f"    ⚠️  {b:35} — {count} posts")
        all_bots.update(bots)
    else:
        print(f"  ✅ {repo_name} — no bots found")

print(f"\nBots to remove: {all_bots}")

Checking for bots in each repo:

  scikit-learn/scikit-learn
    ⚠️  github-actions[bot]                 — 16 posts
    ⚠️  mihara-bot                          — 1 posts
    ⚠️  pranav-bot                          — 2 posts
    ⚠️  scikit-learn-bot                    — 436 posts
  huggingface/transformers
    ⚠️  ParagEkbote                         — 7 posts
    ⚠️  TheSanjBot                          — 1 posts
    ⚠️  github-actions[bot]                 — 799 posts
    ⚠️  tharindubotcalm                     — 1 posts
  pytorch/pytorch
    ⚠️  claude[bot]                         — 5 posts
    ⚠️  github-actions[bot]                 — 105 posts
    ⚠️  pytorch-bot[bot]                    — 1527 posts
    ⚠️  pytorchbot                          — 79 posts
    ⚠️  tolleybot                           — 2 posts
  keras-team/keras
    ⚠️  MagicBOTAlex                        — 3 posts
    ⚠️  github-actions[bot]                 — 817 posts
    ⚠️  google-labs-jules[bot]              — 6 post

***Cell 7b  Bot Detection and Resave***

In [8]:
if not all_bots:
    print("No bots found — CSVs unchanged.")
else:
    print(f"Removing {len(all_bots)} bot account(s): {all_bots}\n")

    for repo_name in REPOS:
        safe_name = SAFE_NAMES[repo_name]
        path      = f"data/processed/{safe_name}_clean.csv"

        if not os.path.exists(path):
            continue

        df     = pd.read_csv(path)
        before = len(df)
        df     = df[~df["author"].isin(all_bots)].copy()
        after  = len(df)
        df.to_csv(path, index=False)

        print(f"  ✅ {repo_name:<35} removed {before - after:>4} rows → {after:,} remaining")

    print("\nAll CSVs re-saved without bots.")

Removing 23 bot account(s): {'aiqing20230305-bot', 'google-labs-jules[bot]', 'MagicBOTAlex', 'ParagEkbote', 'dosubot[bot]', 'TheSanjBot', 'HiperRobot', 'pytorchbot', 'scikit-learn-bot', 'Euni-Bot', 'bobrenze-bot', 'google-ml-butler[bot]', 'langcarl[bot]', 'pytorch-bot[bot]', 'github-actions[bot]', 'mihara-bot', 'pranav-bot', 'langchain-automated-triage[bot]', 'open-swe[bot]', 'YanivBohbot', 'claude[bot]', 'tolleybot', 'tharindubotcalm'}

  ✅ scikit-learn/scikit-learn           removed  455 rows → 11,469 remaining
  ✅ huggingface/transformers            removed  808 rows → 9,073 remaining
  ✅ pytorch/pytorch                     removed 1718 rows → 5,028 remaining
  ✅ keras-team/keras                    removed 1881 rows → 9,324 remaining
  ✅ langchain-ai/langchain              removed  667 rows → 7,656 remaining

All CSVs re-saved without bots.


***Cell 8 Secret Scan***

In [9]:
print("Scanning saved CSVs for residual secrets...\n")

SCAN_PATTERNS = [
    ("OpenAI/OpenRouter key", re.compile(r'sk-[A-Za-z0-9]{20,}')),
    ("AWS Access Key ID",     re.compile(r'AKIA[A-Z0-9]{16}')),
    ("GitHub PAT",            re.compile(r'gh[ps]_[A-Za-z0-9]{36,}')),
]

any_issues = False
for repo_name in REPOS:
    safe_name = SAFE_NAMES[repo_name]
    path = f"data/processed/{safe_name}_clean.csv"
    if not os.path.exists(path):
        continue
    df   = pd.read_csv(path)
    hits = []
    for label, pat in SCAN_PATTERNS:
        for col in ["text", "clean_text"]:
            if col in df.columns:
                count = df[col].dropna().str.contains(pat, regex=True).sum()
                if count > 0:
                    hits.append(f"{label} in '{col}': {count} rows")
                    any_issues = True
    status = "⚠️  " + " | ".join(hits) if hits else "✅ clean"
    print(f"  {repo_name:<40} {status}")

print()
if any_issues:
    print("⚠️  Secrets remain — check SECRET_PATTERNS in Cell 4b.")
else:
    print("✅ All CSVs passed. Safe to push.")

Scanning saved CSVs for residual secrets...

  scikit-learn/scikit-learn                ⚠️  AWS Access Key ID in 'text': 1 rows
  huggingface/transformers                 ✅ clean
  pytorch/pytorch                          ✅ clean
  keras-team/keras                         ✅ clean
  langchain-ai/langchain                   ⚠️  OpenAI/OpenRouter key in 'text': 2 rows | OpenAI/OpenRouter key in 'clean_text': 1 rows

⚠️  Secrets remain — check SECRET_PATTERNS in Cell 4b.


***Cell 8b Targeted Redaction***

In [10]:
TARGETED_FIXES = {
    "scikit-learn_scikit-learn": [
        re.compile(r'AKIA[A-Z0-9]{16}'),
    ],
    "langchain-ai_langchain": [
        re.compile(r'sk-[A-Za-z0-9]{20,}'),
    ],
}

print("Running targeted redaction...\n")

for safe_name, patterns in TARGETED_FIXES.items():
    path = f"data/processed/{safe_name}_clean.csv"
    df   = pd.read_csv(path)

    for pat in patterns:
        for col in ["text", "clean_text"]:
            if col not in df.columns:
                continue
            before = df[col].dropna().str.contains(pat, regex=True).sum()
            if before > 0:
                df[col] = df[col].str.replace(pat, "[REDACTED]", regex=True)
                after   = df[col].dropna().str.contains(pat, regex=True).sum()
                print(f"  ✅ {safe_name} — '{col}': {before} redacted → {after} remaining")

    df.to_csv(path, index=False)
    print(f"  💾 Re-saved: {path}\n")

# Confirm all clear
print("Re-scanning...\n")
all_clear = True
for repo_name in REPOS:
    path = f"data/processed/{SAFE_NAMES[repo_name]}_clean.csv"
    if not os.path.exists(path):
        continue
    df   = pd.read_csv(path)
    hits = []
    for label, pat in [
        ("OpenAI key", re.compile(r'sk-[A-Za-z0-9]{20,}')),
        ("AWS key",    re.compile(r'AKIA[A-Z0-9]{16}')),
        ("GitHub PAT", re.compile(r'gh[ps]_[A-Za-z0-9]{36,}')),
    ]:
        for col in ["text", "clean_text"]:
            if col in df.columns:
                n = df[col].dropna().str.contains(pat, regex=True).sum()
                if n > 0:
                    hits.append(f"{label} in '{col}': {n}")
                    all_clear = False
    status = "⚠️  " + " | ".join(hits) if hits else "✅ clean"
    print(f"  {repo_name:<40} {status}")

print()
print("✅ All clean." if all_clear else "⚠️  Secrets still present — check manually.")

Running targeted redaction...

  ✅ scikit-learn_scikit-learn — 'text': 1 redacted → 0 remaining
  💾 Re-saved: data/processed/scikit-learn_scikit-learn_clean.csv

  ✅ langchain-ai_langchain — 'text': 2 redacted → 0 remaining
  ✅ langchain-ai_langchain — 'clean_text': 1 redacted → 0 remaining
  💾 Re-saved: data/processed/langchain-ai_langchain_clean.csv

Re-scanning...

  scikit-learn/scikit-learn                ✅ clean
  huggingface/transformers                 ✅ clean
  pytorch/pytorch                          ✅ clean
  keras-team/keras                         ✅ clean
  langchain-ai/langchain                   ✅ clean

✅ All clean.


***Cell 9 — Final Verification***

In [11]:
print("FINAL VERIFICATION")
print("=" * 72)
print(f"{'Repo':<35} {'Total':>7} {'Issues':>8} {'Comments':>10} {'Users':>7}")
print("-" * 72)

for repo_name in REPOS:
    safe_name = SAFE_NAMES[repo_name]
    path      = f"data/processed/{safe_name}_clean.csv"

    if not os.path.exists(path):
        print(f"  ❌  {repo_name}")
        continue

    df = pd.read_csv(path)
    print(f"  {repo_name:<35} {len(df):>7,} "
          f"{(df['type']=='issue').sum():>8,} "
          f"{(df['type']=='comment').sum():>10,} "
          f"{df['author'].nunique():>7,}")

print("\nAll files verified. Notebook 2 complete ✅")
print("Next: run Notebook 3 — Network Analysis")

FINAL VERIFICATION
Repo                                  Total   Issues   Comments   Users
------------------------------------------------------------------------
  scikit-learn/scikit-learn            11,469    1,729      9,740   1,856
  huggingface/transformers              9,073    1,990      7,083   2,374
  pytorch/pytorch                       5,028    1,724      3,304     902
  keras-team/keras                      9,324    1,995      7,329   1,654
  langchain-ai/langchain                7,656    1,993      5,663   2,947

All files verified. Notebook 2 complete ✅
Next: run Notebook 3 — Network Analysis
